In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

%cd /content/drive/MyDrive/code/

/content/drive/MyDrive/code


In [3]:
!rm .env

In [31]:
!ls

checkpoint.pt			     main.ipynb
checkpoints			     model_layers.py
data				     __pycache__
eda.ipynb			     requirements-colab.txt
helper_functions.py		     runs
lightweight_googlenet_checkpoint.pt  setup.md
lightweight_googlenet.pth	     test.ipynb


In [5]:
!ls -a

.	       .gitignore			    __pycache__
..	       helper_functions.py		    requirements-colab.txt
checkpoint.pt  .ipynb_checkpoints		    runs
checkpoints    lightweight_googlenet_checkpoint.pt  setup.md
data	       lightweight_googlenet.pth	    test.ipynb
eda.ipynb      main.ipynb
.env	       model_layers.py


In [6]:
import os
from pathlib import Path

from dotenv import load_dotenv
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
from model_layers import LightweightGoogLeNet
from helper_functions import train_loop, test_loop

load_dotenv()

CLASSES = ["Tom", "None", "Jerry", "Both"]
NUM_CLASSES = len(CLASSES)

DATA_DIR = Path(os.getenv("DATA_DIR", "./data"))
IMAGES_DIR = DATA_DIR / "images"
ckpt_path = "checkpoint2.pt"
final_model_path = "lightweight_googlenet2.pth"

generator = torch.Generator().manual_seed(int(os.getenv("RANDOM_SEED", "42")))
batch_size = int(os.getenv("BATCH_SIZE", "256"))
lr = float(os.getenv("LEARNING_RATE", "0.001"))
num_epochs = int(os.getenv("NUM_EPOCHS", "20"))
num_workers = int(os.getenv("NUM_WORKERS", "2"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15),
    transforms.ToTensor()
])

base_dataset = datasets.ImageFolder(root=str(IMAGES_DIR), transform=transform)
dataset_size = len(base_dataset)

train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    base_dataset, [train_size, val_size, test_size], generator=generator)

loader_options = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "pin_memory": device.type == "cuda",
}
train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = torch.utils.data.DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=False, **loader_options)

In [8]:
log_dir = Path(os.getenv("RUNS_DIR", "./runs")) / "tom_and_jerry_experiment5"
writer = SummaryWriter(log_dir=str(log_dir), flush_secs=10)

In [9]:
model = LightweightGoogLeNet(num_classes=NUM_CLASSES).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)
scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None

start_epoch = 0

In [10]:
# Add graph to TensorBoard
try:
    example_inputs = torch.zeros(1, 3, 224, 224, device=device)
    writer.add_graph(model, example_inputs)
except Exception as e:
    print(f"Could not log graph: {e}")

In [23]:
# Run this cell if you want to load a saved checkpoint from disk
if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = checkpoint["epoch"]
    print(f"Checkpoint loaded! Resuming from epoch {start_epoch + 1}.")
else:
    print("No checkpoint found. Starting fresh from epoch 1.")

Checkpoint loaded! Resuming from epoch 41.


In [24]:
num_epochs = 10

In [25]:
print(f"Starting training run: Epochs {start_epoch + 1} to {start_epoch + num_epochs}")
print(f"Learning rate: {scheduler.get_last_lr()}, Batch size: {batch_size}, Device: {device}")

for epoch in range(start_epoch, start_epoch + num_epochs):
    print(f"Epoch {epoch + 1}/{start_epoch + num_epochs} (Global Epoch {epoch + 1})")

    train_loss, train_acc = train_loop(train_loader, model, loss_fn, optimizer, device, scaler)
    val_loss, val_acc = test_loop(val_loader, model, loss_fn, device)

    scheduler.step()

    # Log metrics using global epoch index
    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": train_acc, "val": val_acc}, epoch)
    writer.flush()

# Update global starting point for the next potential cell execution
start_epoch += num_epochs

# Save checkpoint state after the batch of epochs finishes
checkpoint = {
    "epoch": start_epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
}
if scaler:
    checkpoint["scaler_state_dict"] = scaler.state_dict()

torch.save(checkpoint, ckpt_path)
print(f"Run complete. Checkpoint saved at epoch {start_epoch} to {ckpt_path}.")

Starting training run: Epochs 41 to 50
Learning rate: [0.001], Batch size: 32, Device: cuda
Epoch 41/50 (Global Epoch 41)
Training on 120 batches...
Batch 1/120: Loss: 0.0897, Accuracy: 0.9375
Batch 101/120: Loss: 0.1059, Accuracy: 0.9688
Train Loss: 0.1283, Accuracy: 0.9590
Testing on 26 batches...
Test Loss: 0.2240, Accuracy: 0.9464
Epoch 42/50 (Global Epoch 42)
Training on 120 batches...
Batch 1/120: Loss: 0.0899, Accuracy: 0.9688
Batch 101/120: Loss: 0.1092, Accuracy: 0.9688
Train Loss: 0.0942, Accuracy: 0.9695
Testing on 26 batches...
Test Loss: 0.1932, Accuracy: 0.9525
Epoch 43/50 (Global Epoch 43)
Training on 120 batches...
Batch 1/120: Loss: 0.0305, Accuracy: 1.0000
Batch 101/120: Loss: 0.1392, Accuracy: 0.9688
Train Loss: 0.0988, Accuracy: 0.9656
Testing on 26 batches...
Test Loss: 0.1920, Accuracy: 0.9488
Epoch 44/50 (Global Epoch 44)
Training on 120 batches...
Batch 1/120: Loss: 0.1085, Accuracy: 0.9688
Batch 101/120: Loss: 0.1510, Accuracy: 0.9688
Train Loss: 0.0822, Accura

In [26]:
# Run this cell when you are fully done training
writer.close()
torch.save(model.state_dict(), final_model_path)

print(f"SummaryWriter closed. Final model weights saved to '{final_model_path}'.")

SummaryWriter closed. Final model weights saved to 'lightweight_googlenet2.pth'.


In [27]:
test_loss, test_acc = test_loop(test_loader, model, loss_fn, device)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_acc}")

Testing on 26 batches...
Test Loss: 0.3157, Accuracy: 0.9356
Test Loss: 0.3157422955935964, Test Accuracy: 0.9356014580801945
